In [26]:
# crawl_test.py => 독립적인 모듈
# vscode, cursor, pychrm, elipse
# import crawl_test
# 실행여부와 상관없이, 바로 모듈 실행

import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin
import re

BASE = "https://books.toscrape.com/"

# Refactoring => 1개의 긴 문장 코드를 기능별 여러개의 코드로 나눠서 작성, 관리하는 프로그래밍 기법
# 1개의 긴 문장 코드 => 너무 길다보니 버그 발생 => 어디에서 어떤 버그 모름
# 코드 1개의 실행을 위해서 사용해야하는 리스크 높음 => 코드 분산 => 메모리 스레드 => 멀티스레드 & 싱글스레드 => 운영체제

def parse_book(card, category) :
    title = card.h3.a["title"].strip()
    product_url = urljoin(BASE, card.h3.a["href"])
    price_text = card.select_one(".price_color").get_text(strip=True)
    price_num = re.sub(r'[^0-9\.]', "", price_text)
    price = float(price_num) if price_num else None
    stock_txt = card.select_one(".instock.availability").get_text(strip=True)
    stock = "In stock" in stock_txt
    rating_word = card.select_one(".star-rating")["class"][1]
    rating_map = {
        "One" : 1,
        "Two" : 2,
        "Three" : 3,
        "Four" : 4,
        "Five" : 5
    }
    rating = rating_map.get(rating_word)

    return {
        "title" : title,
        "price" : price,
        "stock" : stock,
        "rating" : rating,
        "category" : category,
        "product_url" : product_url
    }
    


def crawl_category(cat_url, max_pages = 2) :
    data = []
    next_url = cat_url
    for i in range(max_pages) :
        html = requests.get(next_url, timeout=10).text
        soup = BeautifulSoup(requests.get(BASE, timeout=10).text, "html.parser")
        category = soup.select_one(".page-header.action h1")
        category = category.get_text(strip=True).strip() if category else "Unknown" # 3항 조건 연산자
        for card in soup.select("ol.row > li.col-xs-6.col-sm-4.col-md-3.col-lg-3") :
            data.append(parse_book(card, category))
        nxt = soup.select_one("li.next a")
        if not nxt :
            break
        next_url = urljoin(next_url, nxt["href"])
        time.sleep(0.5)
        
        
    return data

def crawl_few_categories() :
    html = requests.get(BASE, timeout=10).text
    soup = BeautifulSoup(requests.get(BASE, timeout=10).text, "html.parser")
    cat_links = [urljoin(BASE, a["href"]) for a in soup.select(".side_categories > ul > li > ul > li > a")[:3]]


    all_rows = []
    for link in cat_links :
        all_rows.extend(crawl_category(link))
    return all_rows


if __name__ == "__main__":
    rows = crawl_few_categories()   # ❗ 함수 호출 괄호 추가
    print(f"Collected : {len(rows)} rows")


Collected : 120 rows


In [14]:
list_test = [1, 2, 3, 4]
print(dir(list_test))

['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']


In [ ]:
from pymongo import MongoClient

def save_mongo(rows, db_name="playground", col_name="books") :
    USER_RAW = "sagejin99_db_user"
    PASS_RAW = "%40%21dydwls900" 
    uri = (
        f"mongodb+srv://{USER_RAW}:{PASS_RAW}"
        f"@cluster0.hnhxaft.mongodb.net/?retryWrites=true&w=majority&appName=Cluster0"
    )
    client = MongoClient(uri, serverSelectionTimeoutMS=10000)
    col = client[db_name][col_name]

    for doc in rows :
        col.update_one(
            {
                "title": doc["title"],
                "product_url": doc["product_url"],
                "price": doc["price"],
            }
        )
    print(f"MongoDB : {len(rows)} rows")